# M3 訓練資料：恢復本朗讀＋生命讀經 → 5–30 秒片段（需要 GPU）

這本筆記本把雲端硬碟 `twgbr-archive` 裏的**整章／整篇錄音**，用 Whisper large-v3-turbo 找出每個字的時間，和恢復本／生命讀經的**原文**對齊，切成 5–30 秒的片段（16 kHz 單聲道 WAV），標籤用原文：

- 恢復本朗讀：只留對齊 CER ≤10% 的片段；生命讀經：只留 ≤15% 的片段（講員照稿讀的部分，插話不收）。對齊 CER 以讀音（無聲調拼音）計算：寫法不同但讀音相同（裡／裏、積澱／基甸）不算錯，標籤一律是原文。
- 不收：開發集（`bible-dev-v1` 的 20 章、`life-study-dev-v1` 的 16 篇）、MEET、有多個音檔的篇。
- 約 2% 的章／篇整篇放 validation，不和 train 共用錄音。
- 結果存在 `MyDrive/meeting-live-captions/data/twgbr-v1/`：`audio/`（片段）、`manifests/train.jsonl`、`validation.jsonl`、`reports/data-report.md`。**只限內部使用，不公開、不分享資料夾。**

**怎麼跑**

1. 「執行階段 → 變更執行階段類型 → T4 GPU」→ 儲存。
2. **第一次先試跑**（第 4 格 `TRIAL = True`，每種來源 5 個錄音，約 10–20 分鐘）→「全部執行」→ 最後一格會播放幾段片段並顯示標籤，聽聽看對不對。
3. 試跑沒問題，把 `TRIAL` 改成 `False` 再「全部執行」。完整約需 6–8 小時 GPU：每次最多跑 `MAX_MINUTES` 分鐘，Colab 斷線或時間到就**重新全部執行**，會接著做（已完成的不重做）。
4. 每次跑完回覆「直接讀」，我會讀 `reports/data-report.md`。

雲端硬碟空間：片段約每小時 115 MB；完整約 10–15 GB（以報告為準）。不用 Cloudflare 額度。

In [ ]:
# 第 1 格：連接雲端硬碟、找到資料、確認 GPU
import os, subprocess, shutil, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'recovery-version' in dirs and 'life-study' in dirs]
if not found:
    raise RuntimeError('找不到 twgbr-archive（含 recovery-version 與 life-study）')
ARCHIVE = found[0]
OUTPUT = ROOT/'meeting-live-captions/data/twgbr-v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else ''
free = shutil.disk_usage(ROOT).free/1e9
print('資料：', ARCHIVE, '｜輸出：', OUTPUT, '｜GPU：', gpu or '沒有', f'｜雲端硬碟可用約 {free:.0f} GB')
if not gpu:
    raise RuntimeError('需要 GPU：執行階段 → 變更執行階段類型 → T4 GPU')

In [ ]:
# 第 2 格：取得固定版本的程式與套件（含 GPU 函式庫，約 2–3 分鐘）
COMMIT = '089e2c2698ea4007c747e3fe3606ee639289b409'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'ctranslate2==4.8.3', 'av==16.0.1',
                'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7', 'beautifulsoup4==4.13.4', 'pypinyin==0.55.0',
                # CTranslate2 needs CUDA 12 cuBLAS + cuDNN 9; newer Colab images do not ship libcublas.so.12.
                'nvidia-cublas-cu12==12.9.1.4', 'nvidia-cudnn-cu12==9.28.0.42'], check=True)
import importlib, importlib.util
importlib.invalidate_caches()
lib_dirs = [str(Path(location)/'lib') for module in ('nvidia.cublas', 'nvidia.cudnn')
            for location in importlib.util.find_spec(module).submodule_search_locations]
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('程式版本', COMMIT[:12])

In [ ]:
# 第 3 格：下載固定版本的 Whisper large-v3-turbo 並核對 SHA-256（約 1.6GB）
import hashlib
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('deepdml/faster-whisper-large-v3-turbo-ct2', revision='4df90f75321148c3a29a9e2351b7ddf8f5b115a8',
                              allow_patterns=['config.json', 'model.bin', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json'])
digest = hashlib.sha256()
with open(Path(MODEL_DIR)/'model.bin', 'rb') as stream:
    for block in iter(lambda: stream.read(1 << 24), b''):
        digest.update(block)
if digest.hexdigest() != 'e76620f83d5f5b69efd3d87e3dc180c1bd21df9fbebacfd4335e5e1efcc018da':
    raise RuntimeError('模型檔 SHA-256 不符，已停止')
print('模型已核對：', MODEL_DIR)

In [ ]:
# 第 4 格：設定與計畫（列出要用的章／篇、排除項目與分割）
TRIAL = True               # 第一次 True：每種來源只做 5 個錄音。試跑沒問題再改 False
LIFE_STUDY_MESSAGES = 120  # 生命讀經用幾篇（每篇約 30–45 分鐘音訊）
MAX_MINUTES = 150          # 這次最多辨識幾分鐘，時間到會停；重新執行會接著做
def step(*args):
    command = [sys.executable, '-u', str(REPO/'scripts/build_twgbr_training_data.py'), *args, '--archive', str(ARCHIVE), '--output', str(OUTPUT)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('失敗，請看上面的錯誤訊息')
step('plan', '--life-study-messages', str(LIFE_STUDY_MESSAGES))

In [ ]:
# 第 5 格：Whisper 找出每個字的時間（GPU；已做過的錄音會略過）
step('transcribe', '--model-dir', MODEL_DIR, '--model-label', 'whisper-large-v3-turbo (deepdml ct2 4df90f7)',
     '--max-minutes', str(MAX_MINUTES), *(['--trial', '5'] if TRIAL else []))

In [ ]:
# 第 6 格：對齊原文、切成片段、寫 manifest 與資料報告
step('segment')
print('\n報告：', OUTPUT/'reports/data-report.md')

In [ ]:
# 第 7 格：抽聽幾段（聽聽看標籤和聲音是否一致）
import json, random
from IPython.display import Audio, display
rows = [json.loads(line) for name in ('train', 'validation') for line in (OUTPUT/f'manifests/{name}.jsonl').read_text(encoding='utf-8').splitlines()]
random.seed(len(rows))
for row in random.sample(rows, min(6, len(rows))):
    print(f"{row['id']}｜{row['book_name']}｜{row['duration']:.1f} 秒｜對齊 CER {row['alignment_cer']:.1%}")
    print('  標籤：', row['text'])
    display(Audio(str(OUTPUT/row['audio_filepath'])))